In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt
from scipy.integrate import *
import torch
from bh_horizons import *

In [ ]:
def xi_integral(theta_amp, theta_phase, k, rho_drho_func, omega_domega_func):

    V_grid = np.linspace(0, 1, 5001)
    _, domega_grid = omega_domega_func(V_grid, theta_phase)

    def domega_fast(V):
        return np.interp(np.asarray(V), V_grid, domega_grid)

    def f(V, y):
        xi, xip = y
        rho, drho = rho_drho_func(V, theta_amp, k=k)
        domega = domega_fast(V)

        return [xip, -xi * (drho**2 + (domega * rho)**2)]

    sol = solve_ivp(f, t_span=(1.0, 0.0), y0=[1.0, 0.0], t_eval=V_grid[::-1],method="DOP853", rtol=1e-10, atol=1e-12)

    xi_vals = sol.y[0][::-1]
    xip_vals = sol.y[1][::-1]

    rho_vals = rho_drho_func(V_grid, theta_amp, k=k)[0]
    domega_vals = domega_fast(V_grid)

    I = simpson(xi_vals**2 * rho_vals**2 * domega_vals, x=V_grid)

    return xi_vals, xip_vals, I

In [3]:
def solve_Phi_wave(theta_1, theta_2, r, r_U, r_V, Q, A_U, eM, m, rho_drho_func, omega_domega_func, k=1):
    
    V_grid = np.linspace(0.0, 1.0, len(r))
    rho, drho = rho_drho_func(V_grid, theta_1, k=k)
    omega, domega = omega_domega_func(V_grid, theta_2)

    phase = np.exp(-1j * omega)

    Phi = rho * phase
    Phi_V = (drho - 1j * rho * domega) * phase

    integrand = (
        -r_U * Phi_V
        + 1j * eM * Q * Phi / (4.0 * r)
        - 1j * eM * A_U * r_V * Phi
        - 1j * eM * A_U * r * Phi_V
        - 0.25 * m**2 * r * Phi
    )

    Phi_U_1 = simpson(integrand, x=V_grid) / r[-1]

    return Phi_U_1.real, Phi_U_1.imag

In [ ]:
def loss_C1(theta_1, theta_2, eM, q, rho_drho_func, omega_domega_func, lam=0.0, m_over_e=0.0, w_Q=1.0, w_1=1.0, w_penalty=1000.0, delta_pen=1e-4):
    xi_vals, xip_vals, _ = xi_integral(theta_1, theta_2, k=1, rho_drho_func=rho_drho_func, omega_domega_func=omega_domega_func)

    if not np.all(np.isfinite(xi_vals)):
        return 1e12

    inf_xi = np.min(xi_vals)

    if inf_xi <= 1e-8:
        return 1e6 + 1e6 * (1e-8 - inf_xi)**2

    out = var_values_general(
        theta_1=theta_1,
        theta_2=theta_2,
        k=1,
        xi_vals=xi_vals,
        xip_vals=xip_vals,
        q=q,
        lam=lam,
        m_over_e=m_over_e,
        eM=eM,
        phase_fn=omega_domega_func,
    )

    V = np.linspace(0.0, 1.0, len(xi_vals))

    r = out["r_plus"] * xi_vals
    r_V = out["r_plus"] * xip_vals
    r_U = out["rU_vals"]
    Q = out["Q_vals"]

    A_U = AU_values(V, r, Q)

    re_phi_u, im_phi_u = solve_Phi_wave(
        theta_1=theta_1,
        theta_2=theta_2,
        r=r,
        r_U=r_U,
        r_V=r_V,
        Q=Q,
        A_U=A_U,
        eM=eM,
        m=out["m"],
        omega_domega_func=omega_domega_func,
    )

    penalty = w_penalty * (
        penalty_xi(
            xi_vals,
            delta=delta_pen,
        )
        + penalty_rU(
            r_U,
            delta=delta_pen,
        )
    )

    return (
        w_Q * out["charge_residual"]**2
        + w_1 * (
            re_phi_u**2
            + im_phi_u**2
        )
        + penalty
    )